In [1]:
%pip install -q evaluate

In [2]:
import pandas as pd
import torch
from pathlib import Path
print("GPU available:", torch.cuda.is_available())

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from datasets import Dataset

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments

from evaluate import load as load_metric

import numpy as np

import re
import string

from transformers import set_seed
set_seed(42)

GPU available: True


In [3]:
import sys
import transformers

print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.13.16
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
CUDA: 13.0


# Load dataset

Loading dataset, and setup a tokenizer.

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


saved_model_exists = all(
    Path("multilingual-qa", filename).is_file()
    for filename in ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]
)
model_source = "multilingual-qa" if saved_model_exists else "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(
    model_source, use_fast=True, local_files_only=saved_model_exists
)


README.md:   0%|          | 0.00/9.88k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B / 6.88MB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 1.38MB            

validation.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/4 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [5]:
def prepare_examples(examples):
    tokens = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    offsets = tokens.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, example_number in enumerate(example_numbers):
        cls_position = tokens["input_ids"][i].index(tokenizer.cls_token_id)
        start_position = cls_position
        end_position = cls_position

        if examples["answerable"][example_number]:
            answer_start = int(examples["answer_start"][example_number])
            answer_end = answer_start + len(examples["answer"][example_number])
            context_tokens = [
                j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
            ]
            if (context_tokens
                and offsets[i][context_tokens[0]][0] <= answer_start
                and offsets[i][context_tokens[-1]][1] >= answer_end):
                answer_tokens = [
                    j for j in context_tokens
                    if offsets[i][j][0] < answer_end
                    and offsets[i][j][1] > answer_start
                ]
                if answer_tokens:
                    start_position = answer_tokens[0]
                    end_position = answer_tokens[-1]

        start_positions.append(start_position)
        end_positions.append(end_position)

    tokens["start_positions"] = start_positions
    tokens["end_positions"] = end_positions
    return tokens

model = AutoModelForQuestionAnswering.from_pretrained(
    model_source, local_files_only=saved_model_exists
)

if saved_model_exists:
    training_args = TrainingArguments(
        output_dir="multilingual-qa",
        fp16=True,
        per_device_eval_batch_size=8,
        seed=42,
        full_determinism=True,
        report_to="none",
    )
    trainer = Trainer(model=model, args=training_args)
    print("Loaded saved model. Skipping training.")

else:
    train_examples = Dataset.from_pandas(
        df_train_filtered, preserve_index=False
    )
    train_features = train_examples.map(
        prepare_examples,
        batched=True,
        remove_columns=train_examples.column_names,
    )

    training_args = TrainingArguments(
        output_dir="multilingual-qa",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=8,
        fp16=True,
        learning_rate=3e-5,
        seed=42,
        full_determinism=True,
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_features,
    )
    trainer.train()

    trainer.save_model("multilingual-qa")
    tokenizer.save_pretrained("multilingual-qa")
    print("Saved final model and tokenizer.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Requested unified CUBLASLT workspace size of 1048576 bytes exceeds CUBLAS workspace size of 131072 bytes. Please increase CUBLAS workspace size via CUBLAS_WORKSPACE_CONFIG or decrease requested CUBLASLT_WORKSPACE_SIZE. Otherwise CUBLASLT workspace size will be limited to the CUBLAS workspace size. (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:228.)
  return F.linear(input, self.weight, self.bias)


Step,Training Loss
500,2.146483
1000,1.423250
1500,1.125982
2000,0.764617


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model and tokenizer.


In [6]:
validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)

def prepare_validation(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Local example indices within this preprocessing batch.
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        # Only context tokens can belong to an extracted answer.
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

# One batch makes example_index refer to the whole validation dataset.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)

# Metadata is needed for decoding, but must not be passed to the model.
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = trainer.predict(model_inputs).predictions

n = len(validation_examples)
null_scores = np.full(n, np.inf)
span_scores = np.full(n, -np.inf)
best_answers = [""] * n

# A fixed decoding choice: number of WordPiece tokens, not words.
MAX_ANSWER_LENGTH = 30

for feature_index, feature in enumerate(validation_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]

    cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index],
        float(starts[cls_index] + ends[cls_index]),
    )

    # Search all valid context spans up to MAX_ANSWER_LENGTH.
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue

        for end in range(
            start, min(start + MAX_ANSWER_LENGTH, len(offsets))
        ):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue

            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                context = validation_examples[example_index]["context"]
                best_answers[example_index] = context[
                    start_offset[0]:end_offset[1]
                ]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()

no_answer_scores = null_scores - span_scores

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [7]:
scores = np.asarray(no_answer_scores)
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)

thresholds = []

# First, test a cutoff that predicts no answer for everyone.
threshold_below_all_scores = scores.min() - 1
thresholds.append(threshold_below_all_scores)

# np.unique returns the distinct scores in ascending order.
unique_scores = np.unique(scores)

# Then test each score as a cutoff.
for score in unique_scores:
    thresholds.append(score)

best_threshold = thresholds[0]
best_macro_f1 = -1

for threshold in thresholds:
    predicted_answerable = scores <= threshold

    macro_f1 = f1_score(
        gold_answerable,
        predicted_answerable,
        labels=[False, True],
        average="macro",
        zero_division=0,
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

NO_ANSWER_THRESHOLD = best_threshold

print("Threshold:", NO_ANSWER_THRESHOLD)
print("Validation macro-F1:", best_macro_f1)

Threshold: -0.6354904174804688
Validation macro-F1: 0.8319959271739921


In [8]:
predicted_answers = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, scores)
]

In [9]:
squad_metric = load_metric("squad_v2")

evaluation = df_validation_filtered.copy()
evaluation["prediction"] = predicted_answers
evaluation["pred_answerable"] = evaluation["prediction"] != ""

results = []
for language in ["overall", "ar", "ko", "te"]:
    group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
    formatted_predictions = []
    gold = []

    for example_id, example in group.iterrows():
        formatted_predictions.append({
            "id": str(example_id),
            "prediction_text": example["prediction"],
            # Required field: encode the already-made abstention decision.
            # This is a binary indicator, not a calibrated probability.
            "no_answer_probability": float(not example["pred_answerable"]),
        })
        gold.append({
            "id": str(example_id),
            "answers": {
                "text": [example["answer"]] if example["answerable"] else [],
                "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
            },
        })

    squad_results = squad_metric.compute(
        references=gold, predictions=formatted_predictions
    )
    report = classification_report(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], target_names=["unanswerable", "answerable"],
        output_dict=True, zero_division=0,
    )

    row = {
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": squad_results["exact"],
        "SQuAD_v2_token_F1": squad_results["f1"],
        "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
    }
    for label in ["unanswerable", "answerable"]:
        row[f"{label}_P"] = 100 * report[label]["precision"]
        row[f"{label}_R"] = 100 * report[label]["recall"]
        row[f"{label}_F1"] = 100 * report[label]["f1-score"]
    results.append(row)

results_table = pd.DataFrame(results)
display(results_table.round(2))

,language,N,SQuAD_v2_EM,SQuAD_v2_token_F1,answerability_macro_F1,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1
0,overall,1155,58.27,66.44,83.20,70.00,72.56,71.26,95.43,94.85,95.14
1,ar,415,60.72,69.49,94.28,84.75,96.15,90.09,99.44,97.52,98.47
2,ko,356,59.83,69.02,83.91,62.50,78.95,69.77,98.80,97.33,98.06
3,te,384,54.17,60.74,73.88,62.07,58.06,60.00,86.87,88.66,87.76


In [11]:
def normalize_answer(text):
    text = text.lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

examples = evaluation.copy()
examples["gold_answer"] = examples["answer"].where(
    examples["answerable"], ""
)

errors = examples[
    examples["prediction"].map(normalize_answer)
    != examples["gold_answer"].map(normalize_answer)
]

selected_errors = errors.groupby("lang", sort=False).head(4)

unanswerable_errors = examples[
    (~examples["answerable"]) & (examples["prediction"] != "")
]
extra_errors = unanswerable_errors.groupby("lang", sort=False).head(1)

extra_errors = extra_errors[
    ~extra_errors.index.isin(selected_errors.index)
]
selected_errors = pd.concat([selected_errors, extra_errors])

print("Selected errors:", len(selected_errors))
print(selected_errors["lang"].value_counts())

for index, row in selected_errors.iterrows():
    print(f"\nExample {index} ({row['lang']})")
    print("Question:", row["question"])
    print("Context:", row["context"])
    print("Answerable:", row["answerable"])
    print("Gold answer:", repr(row["gold_answer"]))
    print("Prediction:", repr(row["prediction"]))

Selected errors: 15
lang
te    5
ko    5
ar    5
Name: count, dtype: int64

Example 0 (te)
Question: ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?
Context: Portland is the largest city in the U.S. state of Oregon and the seat of Multnomah County. It is a major port in the Willamette Valley region of the Pacific Northwest, at the confluence of the Willamette and Columbia rivers. As of 2017, Portland had an estimated population of 647,805, making it the 26th-largest city in the United States, and the second-most populous in the Pacific Northwest (after Seattle). Approximately 2.4 million people live in the Portland metropolitan statistical area (MSA), making it the 25th most populous MSA in the United States. Its Combined Statistical Area (CSA) ranks 18th-largest with a population of around 3.2 million. Approximately 60% of Oregon's population resides within the Portland metropolitan area.
Answerable: True
Gold answer: 'Portland'
Prediction: 'Seattle'

Example 2 (te)
Question: కలరా వ్యాధిని మ